# TP 5 — Classer l'occupation du sol à Yamoussoukro

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP5_Yamoussoukro_RF_validation_spatiale.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 5 · KOUADIO Kouao Laurent

Objectifs :
1. Construire un jeu d'apprentissage : pixels Sentinel-2 + classes
   ESA WorldCover.
2. Entraîner une **forêt aléatoire**.
3. Comparer la validation **au hasard** et la validation **par blocs
   spatiaux**.
4. Observer le compromis **biais–variance**.
5. Bonus : **k-means** et **ACP**.

Durée : environ 90 minutes. Connexion Internet nécessaire.
Pixels de 60 m pour que la lecture reste courte.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac rioxarray

## 1. Imports et paramètres

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import rioxarray
import xarray as xr
from matplotlib.colors import ListedColormap
from odc.stac import configure_rio, load
from pystac_client import Client
from rasterio.enums import Resampling
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             f1_score)
from sklearn.model_selection import (GroupKFold, KFold,
                                     cross_val_predict)
from sklearn.preprocessing import StandardScaler

configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10",
              GDAL_HTTP_RETRY_DELAY="2")
GRAINE = 0                          # reproductibilité
BBOX = [-5.40, 6.70, -5.10, 7.00]   # Yamoussoukro
CRS, RES = "EPSG:32630", 60         # UTM 30N, pixels de 60 m
CLASSES = {10: ("Arbres", "#006400"), 20: ("Arbustes", "#ffbb22"),
           30: ("Herbacées", "#ffff4c"), 40: ("Cultures", "#f096ff"),
           50: ("Bâti", "#fa0000"), 60: ("Sol nu", "#b4b4b4"),
           80: ("Eau", "#0064c8"), 90: ("Zones humides", "#0096a0")}

## 2. Composite Sentinel-2 de saison sèche

Médiane des pixels clairs de décembre 2020 à février 2021. Ces scènes
sont antérieures à 2022 : pas de décalage d'offset (voir S4).

In [ ]:
cat = Client.open("https://earth-search.aws.element84.com/v1")
items = cat.search(collections=["sentinel-2-l2a"], bbox=BBOX,
                   datetime="2020-12-01/2021-02-28",
                   query={"eo:cloud_cover": {"lt": 40}}
                   ).item_collection()
BANDES = ["blue", "green", "red", "nir", "swir16", "swir22"]
print(len(items), "scènes")

t0 = time.time()
ds = load(items, bands=BANDES + ["scl"], bbox=BBOX, crs=CRS,
          resolution=RES, chunks={}, groupby="solar_day")
clair = ds["scl"].isin([4, 5, 6, 11])
comp = xr.Dataset({b: ds[b].where(clair) * 1e-4 for b in BANDES})
comp = comp.median("time").compute()
print(f"composite {comp.sizes['y']} x {comp.sizes['x']} "
      f"en {time.time() - t0:.0f} s")

## 3. Étiquettes ESA WorldCover 2021

In [ ]:
url = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/"
       "2021/map/ESA_WorldCover_10m_2021_v200_N06W006_Map.tif")
wc = rioxarray.open_rasterio(url).squeeze(drop=True)
wc = wc.rio.clip_box(*BBOX, crs="EPSG:4326")
# 10 m -> 60 m : la classe majoritaire (mode) de chaque pixel
lab = wc.rio.reproject_match(comp["red"].rio.write_crs(CRS),
                             resampling=Resampling.mode).values
cl, n = np.unique(lab, return_counts=True)
for c, k in zip(cl, n):
    print(c, CLASSES.get(int(c), ("?",))[0], k)

**Question 1.** Quelles classes dominent ? WorldCover est une carte
produite par un modèle : que cela implique-t-il pour nos scores ?

## 4. Variables et échantillons

In [ ]:
F = {b: comp[b].values for b in BANDES}
F["NDVI"] = (F["nir"] - F["red"]) / (F["nir"] + F["red"])
F["NDWI"] = (F["green"] - F["nir"]) / (F["green"] + F["nir"])
F["NDBI"] = (F["swir16"] - F["nir"]) / (F["swir16"] + F["nir"])
NOMS = list(F)
X_tout = np.stack([F[k] for k in NOMS], axis=-1)
H, W = lab.shape
valide = np.isfinite(X_tout).all(-1) & np.isin(lab, list(CLASSES))

rng = np.random.default_rng(GRAINE)
garder = [c for c in CLASSES if (lab[valide] == c).sum() >= 300]
idx = np.concatenate([
    rng.choice(np.flatnonzero((lab == c).ravel() & valide.ravel()),
               size=min(300, int(((lab == c) & valide).sum())),
               replace=False) for c in garder])
X = X_tout.reshape(-1, len(NOMS))[idx]
y = lab.ravel()[idx]
lig, col = np.unravel_index(idx, (H, W))
groupes = (lig // 50) * 1000 + col // 50   # blocs de 50 px = 3 km
print(X.shape, "échantillons,", np.unique(groupes).size, "blocs")

## 5. Forêt aléatoire : au hasard ou par blocs ?

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                            random_state=GRAINE)
kf = KFold(5, shuffle=True, random_state=GRAINE)
gk = GroupKFold(5)
p_hasard = cross_val_predict(rf, X, y, cv=kf)
p_blocs = cross_val_predict(rf, X, y, cv=gk, groups=groupes)
for nom, p in [("hasard", p_hasard), ("blocs", p_blocs)]:
    print(f"{nom:7s} précision {accuracy_score(y, p):.3f}  "
          f"F1 {f1_score(y, p, average='macro'):.3f}")

**Question 2.** Quelle validation donne le meilleur score ? Laquelle
faut-il annoncer dans un rapport ? Pourquoi ?

In [ ]:
noms = [CLASSES[c][0] for c in garder]
ConfusionMatrixDisplay.from_predictions(
    y, p_blocs, labels=garder, display_labels=noms,
    normalize="true", values_format=".2f", cmap="Greens",
    xticks_rotation=45)
plt.show()

**Question 3.** Quelles classes se confondent ? Proposez une
explication physique.

## 6. Biais et variance

In [ ]:
profs = [1, 2, 4, 8, 16, None]
train, test = [], []
for d in profs:
    m = RandomForestClassifier(n_estimators=100, max_depth=d,
                               n_jobs=-1, random_state=GRAINE)
    train.append(accuracy_score(y, m.fit(X, y).predict(X)))
    test.append(accuracy_score(y, cross_val_predict(
        m, X, y, cv=gk, groups=groupes)))
etiq = [str(d) for d in profs[:-1]] + ["max"]
plt.plot(etiq, train, "o-", label="entraînement")
plt.plot(etiq, test, "o-", label="test (blocs)")
plt.xlabel("profondeur maximale")
plt.ylabel("précision")
plt.legend()
plt.show()

**Question 4.** À partir de quelle profondeur le modèle
sur-apprend-il ?

## 7. La carte

In [ ]:
rf.fit(X, y)
plat = X_tout.reshape(-1, len(NOMS))
pred = np.zeros(H * W, dtype=int)
pred[valide.ravel()] = rf.predict(plat[valide.ravel()])
pred = pred.reshape(H, W)
print(f"accord avec WorldCover : {(pred[valide] == lab[valide]).mean():.3f}")

cmap = ListedColormap([CLASSES[c][1] for c in garder])
def indexer(a):
    out = np.full(a.shape, np.nan)
    for k, c in enumerate(garder):
        out[a == c] = k
    return out
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
for ax, a, t in [(axs[0], lab, "WorldCover"),
                 (axs[1], pred, "forêt aléatoire")]:
    ax.imshow(indexer(a), cmap=cmap, vmin=0, vmax=len(garder) - 1,
              interpolation="nearest")
    ax.set_title(t)
    ax.set_axis_off()
plt.show()
print(dict(zip(NOMS, rf.feature_importances_.round(3))))

## 8. Bonus : ACP et k-means

In [ ]:
Z = StandardScaler().fit_transform(plat[valide.ravel()])
acp = PCA().fit(Z)
print("variance expliquée :", acp.explained_variance_ratio_.round(3))

km = KMeans(6, n_init=4, random_state=GRAINE).fit(Z)
groupes_km = np.full(H * W, np.nan)
groupes_km[valide.ravel()] = km.labels_
plt.imshow(groupes_km.reshape(H, W), cmap="tab10",
           interpolation="nearest")
plt.axis("off")
plt.show()

**Question 5.** Combien de composantes faut-il pour garder 95 % de
la variance ? Nommez trois des six groupes k-means en les comparant à
la carte WorldCover.

## À rendre

Le notebook exécuté avec vos réponses aux questions 1 à 5.